<a href="https://colab.research.google.com/github/aannddrree/disciplinaIA/blob/main/sistema_especialista_rule_engine.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Sistema especialista com Python e `rule-engine`

Este notebook implementa um sistema especialista didático para diagnóstico inicial de um notebook que não liga. Ele contém:

- regras declarativas;
- memória de trabalho;
- encadeamento para frente até o ponto fixo;
- trilha de explicação;
- detecção de conflitos;
- identificação simples de fatos ausentes;
- testes automatizados.

> O diagnóstico é apenas um exemplo acadêmico.

## 1. Instalação

A próxima célula instala a biblioteca no mesmo ambiente do kernel. Depois da primeira execução, ela pode ser comentada.

In [1]:
%pip install -q rule-engine


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.0/135.0 kB 3.1 MB/s eta 0:00:00


## 2. Importações e representação das regras

Cada regra possui nome, expressão condicional, conclusão e prioridade. Expressões são avaliadas pela biblioteca sem usar `eval()`.

In [2]:
import rule_engine
from pprint import pprint
from dataclasses import dataclass
from typing import Any

print("rule-engine importado com sucesso")


rule-engine importado com sucesso


In [3]:
REGRAS = [
    {
        "nome": "R1_sem_energia",
        "quando": "energia == false",
        "entao": {"diagnostico": "verificar_fonte"},
        "prioridade": 100,
    },
    {
        "nome": "R2_com_energia_sem_led",
        "quando": "energia == true and led == false",
        "entao": {"diagnostico": "verificar_tela"},
        "prioridade": 80,
    },
    {
        "nome": "R3_inicia_sistema",
        "quando": "energia == true and led == true and inicia == false",
        "entao": {"diagnostico": "verificar_sistema_operacional"},
        "prioridade": 60,
    },
    {
        "nome": "R4_operacional",
        "quando": "energia == true and led == true and inicia == true",
        "entao": {"estado": "operacional"},
        "prioridade": 40,
    },
]

pprint(REGRAS)


[{'entao': {'diagnostico': 'verificar_fonte'},
  'nome': 'R1_sem_energia',
  'prioridade': 100,
  'quando': 'energia == false'},
 {'entao': {'diagnostico': 'verificar_tela'},
  'nome': 'R2_com_energia_sem_led',
  'prioridade': 80,
  'quando': 'energia == true and led == false'},
 {'entao': {'diagnostico': 'verificar_sistema_operacional'},
  'nome': 'R3_inicia_sistema',
  'prioridade': 60,
  'quando': 'energia == true and led == true and inicia == false'},
 {'entao': {'estado': 'operacional'},
  'nome': 'R4_operacional',
  'prioridade': 40,
  'quando': 'energia == true and led == true and inicia == true'}]


## 3. Compilação

As expressões são compiladas uma única vez. Assim, erros de sintaxe aparecem antes do atendimento do primeiro caso.

In [4]:
def compilar(regras):
    compiladas = []
    for regra in regras:
        compiladas.append({
            **regra,
            "condicao": rule_engine.Rule(regra["quando"]),
        })
    return sorted(compiladas, key=lambda r: r["prioridade"], reverse=True)

REGRAS_COMPILADAS = compilar(REGRAS)
print(f"{len(REGRAS_COMPILADAS)} regras compiladas")


4 regras compiladas


## 4. Motor de inferência

O motor percorre as regras por prioridade e adiciona fatos até nenhuma regra produzir mudança. Se duas regras tentarem atribuir valores diferentes ao mesmo fato derivado, o conflito é registrado e o valor anterior é preservado.

In [5]:
@dataclass
class ResultadoInferencia:
    memoria: dict[str, Any]
    trilha: list[dict[str, Any]]
    conflitos: list[dict[str, Any]]

def inferir(fatos, regras=REGRAS_COMPILADAS, max_ciclos=20):
    memoria = dict(fatos)
    trilha = []
    conflitos = []
    conflitos_vistos = set()

    for ciclo in range(1, max_ciclos + 1):
        mudou = False

        for regra in regras:
            try:
                aplicavel = regra["condicao"].matches(memoria)
            except rule_engine.errors.SymbolResolutionError:
                # A regra depende de um fato que ainda não existe.
                aplicavel = False

            if not aplicavel:
                continue

            novos = {}
            for chave, valor in regra["entao"].items():
                if chave in memoria and memoria[chave] != valor:
                    # Fatos observados e conclusões anteriores não são sobrescritos.
                    assinatura = (regra["nome"], chave, repr(memoria[chave]), repr(valor))
                    if assinatura not in conflitos_vistos:
                        conflitos.append({
                            "regra": regra["nome"],
                            "fato": chave,
                            "valor_existente": memoria[chave],
                            "valor_proposto": valor,
                        })
                        conflitos_vistos.add(assinatura)
                elif memoria.get(chave) != valor:
                    novos[chave] = valor

            if novos:
                memoria.update(novos)
                trilha.append({
                    "ciclo": ciclo,
                    "regra": regra["nome"],
                    "condicao": regra["quando"],
                    "conclusoes": novos,
                })
                mudou = True

        if not mudou:
            return ResultadoInferencia(memoria, trilha, conflitos)

    raise RuntimeError("Limite de ciclos atingido; verifique regras cíclicas.")


## 5. Execução de casos

### Caso A — equipamento sem energia

In [6]:
caso_a = {"energia": False, "led": False, "inicia": False}
resultado_a = inferir(caso_a)
print("Memória final:")
pprint(resultado_a.memoria)
print("\nTrilha:")
pprint(resultado_a.trilha)
print("\nConflitos:")
pprint(resultado_a.conflitos)


Memória final:
{'diagnostico': 'verificar_fonte',
 'energia': False,
 'inicia': False,
 'led': False}

Trilha:
[{'ciclo': 1,
  'conclusoes': {'diagnostico': 'verificar_fonte'},
  'condicao': 'energia == false',
  'regra': 'R1_sem_energia'}]

Conflitos:
[]


### Caso B — possui energia, mas o LED não acende

In [7]:
caso_b = {"energia": True, "led": False, "inicia": False}
resultado_b = inferir(caso_b)
pprint(resultado_b.memoria)
pprint(resultado_b.trilha)


{'diagnostico': 'verificar_tela',
 'energia': True,
 'inicia': False,
 'led': False}
[{'ciclo': 1,
  'conclusoes': {'diagnostico': 'verificar_tela'},
  'condicao': 'energia == true and led == false',
  'regra': 'R2_com_energia_sem_led'}]


### Caso C — liga, mas não inicia o sistema operacional

In [8]:
caso_c = {"energia": True, "led": True, "inicia": False}
resultado_c = inferir(caso_c)
pprint(resultado_c.memoria)
pprint(resultado_c.trilha)


{'diagnostico': 'verificar_sistema_operacional',
 'energia': True,
 'inicia': False,
 'led': True}
[{'ciclo': 1,
  'conclusoes': {'diagnostico': 'verificar_sistema_operacional'},
  'condicao': 'energia == true and led == true and inicia == false',
  'regra': 'R3_inicia_sistema'}]


### Caso D — funcionamento normal

In [9]:
caso_d = {"energia": True, "led": True, "inicia": True}
resultado_d = inferir(caso_d)
pprint(resultado_d.memoria)
pprint(resultado_d.trilha)


{'energia': True, 'estado': 'operacional', 'inicia': True, 'led': True}
[{'ciclo': 1,
  'conclusoes': {'estado': 'operacional'},
  'condicao': 'energia == true and led == true and inicia == true',
  'regra': 'R4_operacional'}]


## 6. Explicação legível

In [10]:
def explicar(resultado):
    if not resultado.trilha:
        print("Nenhuma regra produziu um novo fato.")
    for passo in resultado.trilha:
        print(
            f"Ciclo {passo['ciclo']}: {passo['regra']} disparou porque "
            f"[{passo['condicao']}]. Concluiu {passo['conclusoes']}."
        )
    for conflito in resultado.conflitos:
        print(
            f"CONFLITO: {conflito['regra']} propôs "
            f"{conflito['fato']}={conflito['valor_proposto']!r}, "
            f"mas já existia {conflito['valor_existente']!r}."
        )

explicar(resultado_c)


Ciclo 1: R3_inicia_sistema disparou porque [energia == true and led == true and inicia == false]. Concluiu {'diagnostico': 'verificar_sistema_operacional'}.


## 7. Fatos ausentes

Esta função usa uma lista explícita de perguntas do domínio. Em sistemas reais, o esquema de entrada deve declarar tipos, obrigatoriedade e origem de cada fato.

In [11]:
PERGUNTAS = {
    "energia": "O equipamento recebe energia?",
    "led": "O LED indicador acende?",
    "inicia": "O sistema operacional inicia?",
}

def fatos_faltantes(fatos):
    return {chave: pergunta for chave, pergunta in PERGUNTAS.items() if chave not in fatos}

entrada_incompleta = {"energia": True}
pprint(fatos_faltantes(entrada_incompleta))


{'inicia': 'O sistema operacional inicia?', 'led': 'O LED indicador acende?'}


## 8. Demonstração de conflito

Adicionamos deliberadamente uma regra contraditória para verificar se o motor impede sobrescrita silenciosa.

In [12]:
REGRA_CONTRADITORIA = {
    "nome": "RX_contraditoria",
    "quando": "energia == true and led == false",
    "entao": {"diagnostico": "sem_defeito"},
    "prioridade": 10,
}

regras_com_conflito = compilar(REGRAS + [REGRA_CONTRADITORIA])
resultado_conflito = inferir(caso_b, regras_com_conflito)
explicar(resultado_conflito)


Ciclo 1: R2_com_energia_sem_led disparou porque [energia == true and led == false]. Concluiu {'diagnostico': 'verificar_tela'}.
CONFLITO: RX_contraditoria propôs diagnostico='sem_defeito', mas já existia 'verificar_tela'.


## 9. Testes automatizados

A célula abaixo usa apenas `assert`, portanto não depende de `pytest`.

In [13]:
def executar_testes():
    assert inferir(caso_a).memoria["diagnostico"] == "verificar_fonte"
    assert inferir(caso_b).memoria["diagnostico"] == "verificar_tela"
    assert inferir(caso_c).memoria["diagnostico"] == "verificar_sistema_operacional"
    assert inferir(caso_d).memoria["estado"] == "operacional"

    conflito = inferir(caso_b, regras_com_conflito)
    assert conflito.memoria["diagnostico"] == "verificar_tela"
    assert len(conflito.conflitos) == 1

    assert set(fatos_faltantes({"energia": True})) == {"led", "inicia"}
    print("Todos os testes passaram ✅")

executar_testes()


Todos os testes passaram ✅


## 10. Exercícios sugeridos

1. Adicione o fato `carregador_testado` e uma regra correspondente.
2. Acrescente um campo de certeza às conclusões.
3. Faça a política de conflito escolher a regra de maior prioridade.
4. Carregue as regras de um arquivo JSON.
5. Crie uma interface com Streamlit ou uma API com FastAPI.

## Conclusão

A biblioteca realiza a análise e avaliação segura das expressões. O motor construído no notebook fornece memória de trabalho, encadeamento, explicação e tratamento básico de conflitos. A validação das regras por especialistas continua indispensável.